# Macro AI Lakehouse - Silver Layer Purification
Cleans, transforms, conforms, and persists Silver layer relational tables into DuckDB and Apache Parquet format:
- `silver.fact_macro_monthly`
- `silver.dim_labor_occupations`
- `silver.fact_tech_investment`
- `silver.fact_labor_task_decomposition`

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np

# Standardize base and data directory paths
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
BRONZE_DIR = os.path.join(DATA_DIR, "bronze")
SILVER_DIR = os.path.join(DATA_DIR, "silver")
DB_PATH = os.path.join(DATA_DIR, "lakehouse.duckdb")

os.makedirs(SILVER_DIR, exist_ok=True)
con = duckdb.connect(DB_PATH)
con.execute("CREATE SCHEMA IF NOT EXISTS silver;")
print(f"DuckDB database connected: {DB_PATH}")


DuckDB database connected: C:\Users\aabha\macro_ai_lakehouse\data\lakehouse.duckdb


## 1. Conformed Macro Feeds (`silver.fact_macro_monthly`)

In [2]:
# Standardize GPR monthly and aggregate daily market series
gpr_path = [os.path.join(BRONZE_DIR, f) for f in os.listdir(BRONZE_DIR) if "gpr" in f.lower()][0]
raw_gpr = pd.read_excel(gpr_path) if gpr_path.endswith((".xls", ".xlsx")) else pd.read_csv(gpr_path)
df_gpr = raw_gpr.copy()

if "month" in df_gpr.columns:
    df_gpr["month_date"] = pd.to_datetime(df_gpr["month"].astype(str), format="%Y%m", errors="coerce")
    if df_gpr["month_date"].isna().all():
        df_gpr["month_date"] = pd.to_datetime(df_gpr["month"], errors="coerce")
elif "period" in df_gpr.columns:
    df_gpr["month_date"] = pd.to_datetime(df_gpr["period"], errors="coerce")
else:
    df_gpr["month_date"] = pd.to_datetime(df_gpr.iloc[:, 0], errors="coerce")

df_gpr["month_date"] = df_gpr["month_date"].dt.to_period("M").dt.to_timestamp()

cols_to_keep = {"month_date": "month_date"}
for col in df_gpr.columns:
    c_lower = col.lower()
    if c_lower in ["gpr", "gpr_recent", "global_gpr"]:
        cols_to_keep[col] = "gpr_index"
    elif c_lower in ["gprt", "gpr_threats"]:
        cols_to_keep[col] = "gpr_threats"
    elif c_lower in ["gpra", "gpr_acts"]:
        cols_to_keep[col] = "gpr_acts"

df_gpr_clean = df_gpr[list(cols_to_keep.keys())].rename(columns=cols_to_keep).dropna(subset=["month_date"])

def process_daily_market_csv(filename, price_col_prefix):
    path = os.path.join(BRONZE_DIR, filename)
    df = pd.read_csv(path)
    date_col = [c for c in df.columns if "date" in c.lower()][0]
    close_col = [c for c in df.columns if "close" in c.lower() or "adj" in c.lower()][0]
    
    df["Date"] = pd.to_datetime(df[date_col])
    df["month_date"] = df["Date"].dt.to_period("M").dt.to_timestamp()
    df[close_col] = pd.to_numeric(df[close_col], errors="coerce")
    
    monthly = df.groupby("month_date")[close_col].agg(avg="mean", vol="std").reset_index()
    monthly.rename(columns={
        "avg": f"{price_col_prefix}_monthly_avg",
        "vol": f"{price_col_prefix}_monthly_volatility"
    }, inplace=True)
    return monthly

df_brent_monthly = process_daily_market_csv("market_brent_daily.csv", "brent_crude")
df_fx_monthly = process_daily_market_csv("market_eur_usd_daily.csv", "eur_usd")
df_treasury_monthly = process_daily_market_csv("market_treasury_10y_daily.csv", "us_10y_yield")

fact_macro = (
    df_gpr_clean
    .merge(df_brent_monthly, on="month_date", how="inner")
    .merge(df_fx_monthly, on="month_date", how="inner")
    .merge(df_treasury_monthly, on="month_date", how="inner")
    .sort_values("month_date")
    .reset_index(drop=True)
)

con.register("df_fact_macro", fact_macro)
con.execute("""
    CREATE OR REPLACE TABLE silver.fact_macro_monthly AS 
    SELECT * FROM df_fact_macro;
""")

parquet_macro_path = os.path.join(SILVER_DIR, "fact_macro_monthly.parquet").replace("\\", "/")
con.execute(f"COPY silver.fact_macro_monthly TO '{parquet_macro_path}' (FORMAT PARQUET);")
print(f"Created silver.fact_macro_monthly ({len(fact_macro)} records)")


Created silver.fact_macro_monthly (140 records)


## 2. Labor Occupations & Wage Dimensions (`silver.dim_labor_occupations`)

In [3]:
# Clean O*NET occupations, BLS employment/wages, and task intensity scores
occ_files = [f for f in os.listdir(BRONZE_DIR) if "occupation" in f.lower() and not f.endswith(".zip")]
occ_path = os.path.join(BRONZE_DIR, occ_files[0])
sep_occ = "\t" if occ_path.endswith((".tsv", ".txt")) else ","

df_occ = pd.read_csv(occ_path, sep=sep_occ)
df_occ.columns = [str(c).strip().lower() for c in df_occ.columns]
soc_col = [c for c in df_occ.columns if "soc" in c or "code" in c][0]
title_col = [c for c in df_occ.columns if "title" in c][0]

df_occ["soc_code"] = df_occ[soc_col].astype(str).str.extract(r"(\d{2}-\d{4})")[0]
df_occ_clean = df_occ[["soc_code", title_col]].rename(columns={title_col: "occupation_title"}).drop_duplicates(subset=["soc_code"])

# Ingest BLS national wage and employment estimates
bls_files = [f for f in os.listdir(BRONZE_DIR) if ("bls" in f.lower() or "nat" in f.lower()) and not f.endswith(".zip")]
bls_path = os.path.join(BRONZE_DIR, bls_files[0])
df_bls = pd.read_excel(bls_path) if bls_path.endswith((".xlsx", ".xls")) else pd.read_csv(bls_path)
df_bls.columns = [str(c).strip().lower() for c in df_bls.columns]

bls_soc = [c for c in df_bls.columns if c == "occ_code" or "soc" in c][0]
bls_emp = [c for c in df_bls.columns if c in ["tot_emp", "total_employment", "employment"]][0]
bls_wage = [c for c in df_bls.columns if c in ["h_median", "hourly_median", "hourly_median_wage"]][0]

df_bls_clean = pd.DataFrame({
    "soc_code": df_bls[bls_soc].astype(str).str.strip(),
    "total_employment": pd.to_numeric(df_bls[bls_emp].astype(str).str.replace(",", ""), errors="coerce"),
    "hourly_median_wage": pd.to_numeric(df_bls[bls_wage].astype(str).str.replace(",", ""), errors="coerce")
}).dropna(subset=["hourly_median_wage"])

# Aggregate O*NET work activity importance ratings
act_files = [f for f in os.listdir(BRONZE_DIR) if "activit" in f.lower() and not f.endswith(".zip")]
act_path = os.path.join(BRONZE_DIR, act_files[0])
sep_act = "\t" if act_path.endswith((".tsv", ".txt")) else ","

df_act = pd.read_csv(act_path, sep=sep_act)
df_act.columns = [str(c).strip().lower() for c in df_act.columns]
act_soc = [c for c in df_act.columns if "soc" in c or "code" in c][0]
act_val = [c for c in df_act.columns if "data_value" in c or "value" in c or "rating" in c][0]

df_act["soc_code"] = df_act[act_soc].astype(str).str.extract(r"(\d{2}-\d{4})")[0]
df_act[act_val] = pd.to_numeric(df_act[act_val], errors="coerce")
if "scale_id" in df_act.columns:
    df_act = df_act[df_act["scale_id"].astype(str).str.upper() == "IM"]

df_act_summary = df_act.groupby("soc_code")[act_val].mean().reset_index()
df_act_summary.rename(columns={act_val: "onet_task_intensity_score"}, inplace=True)

# Conformed join
dim_labor = (
    df_occ_clean
    .merge(df_bls_clean, on="soc_code", how="inner")
    .merge(df_act_summary, on="soc_code", how="inner")
)

major_soc_map = {
    "11": "Management", "13": "Business & Financial", "15": "Computer & Mathematical",
    "17": "Architecture & Engineering", "19": "Life & Social Science", "21": "Community & Social Service",
    "23": "Legal", "25": "Educational Instruction", "27": "Arts & Media",
    "29": "Healthcare Practitioners", "31": "Healthcare Support", "33": "Protective Service",
    "35": "Food Preparation & Serving", "37": "Building Maintenance", "39": "Personal Care",
    "41": "Sales & Related", "43": "Office & Admin Support", "45": "Farming & Forestry",
    "47": "Construction & Extraction", "49": "Installation & Repair", "51": "Production",
    "53": "Transportation"
}
dim_labor["soc_major_group"] = dim_labor["soc_code"].str[:2].map(major_soc_map).fillna("Other")

con.register("df_dim_labor", dim_labor)
con.execute("""
    CREATE OR REPLACE TABLE silver.dim_labor_occupations AS 
    SELECT 
        soc_code,
        occupation_title,
        soc_major_group,
        total_employment,
        hourly_median_wage,
        ROUND(onet_task_intensity_score, 2) AS onet_task_intensity_score
    FROM df_dim_labor;
""")

parquet_labor_path = os.path.join(SILVER_DIR, "dim_labor_occupations.parquet").replace("\\", "/")
con.execute(f"COPY silver.dim_labor_occupations TO '{parquet_labor_path}' (FORMAT PARQUET);")
print(f"Created silver.dim_labor_occupations ({len(dim_labor)} rows)")


Created silver.dim_labor_occupations (696 rows)


## 3. BEA Tech & R&D Investment Growth (`silver.fact_tech_investment`)

In [4]:
# Ingest quarterly BEA software and R&D CapEx and compute QoQ growth
bronze_fred_path = os.path.join(BRONZE_DIR, "fred_tech_investment.csv").replace("\\", "/")
con.execute(f"""
    CREATE OR REPLACE TABLE silver.fact_tech_investment AS
    WITH raw AS (
        SELECT 
            CAST(date AS DATE) AS quarter_date,
            software_investment_billions,
            rd_investment_billions,
            (software_investment_billions + rd_investment_billions) AS total_ip_investment_billions
        FROM read_csv_auto('{bronze_fred_path}')
        WHERE date IS NOT NULL
    ),
    growth AS (
        SELECT 
            quarter_date,
            software_investment_billions,
            rd_investment_billions,
            total_ip_investment_billions,
            (software_investment_billions - LAG(software_investment_billions, 1) OVER (ORDER BY quarter_date)) 
                / LAG(software_investment_billions, 1) OVER (ORDER BY quarter_date) * 100 AS software_growth_qoq,
            (rd_investment_billions - LAG(rd_investment_billions, 1) OVER (ORDER BY quarter_date)) 
                / LAG(rd_investment_billions, 1) OVER (ORDER BY quarter_date) * 100 AS rd_growth_qoq,
            (total_ip_investment_billions - LAG(total_ip_investment_billions, 1) OVER (ORDER BY quarter_date)) 
                / LAG(total_ip_investment_billions, 1) OVER (ORDER BY quarter_date) * 100 AS total_ip_growth_qoq
        FROM raw
    )
    SELECT * FROM growth
    WHERE quarter_date >= '2000-01-01'
    ORDER BY quarter_date ASC;
""")

parquet_tech_path = os.path.join(SILVER_DIR, "fact_tech_investment.parquet").replace("\\", "/")
con.execute(f"COPY silver.fact_tech_investment TO '{parquet_tech_path}' (FORMAT PARQUET);")
print("Created silver.fact_tech_investment")


Created silver.fact_tech_investment


## 4. O*NET Task Decomposition: Augmentation vs. Drag (`silver.fact_labor_task_decomposition`)

In [5]:
# Decompose O*NET work activities into augmentation potential and administrative drag
work_act_path = os.path.join(BRONZE_DIR, "work_activities.csv").replace("\\", "/")
con.execute(f"""
    CREATE OR REPLACE TABLE silver.fact_labor_task_decomposition AS
    WITH raw_tasks AS (
        SELECT 
            "O*NET-SOC Code" AS onet_soc_code,
            SUBSTRING("O*NET-SOC Code", 1, 7) AS soc_code,
            "Element ID" AS element_id,
            "Element Name" AS element_name,
            "Scale ID" AS scale_id,
            CAST("Data Value" AS DOUBLE) AS data_value
        FROM read_csv_auto('{work_act_path}')
        WHERE "Scale ID" = 'IM'
    ),
    flagged_tasks AS (
        SELECT 
            soc_code,
            element_id,
            element_name,
            data_value,
            CASE 
                WHEN element_id IN ('4.A.2.a.4', '4.A.2.b.1', '4.A.2.b.2', '4.A.4.a.1', '4.A.1.b.1', '4.A.2.a.1') 
                THEN 1 ELSE 0 
            END AS is_augmentation_task,
            CASE 
                WHEN element_id IN ('4.A.3.b.6', '4.A.4.a.2', '4.A.4.a.5', '4.A.4.b.2') 
                THEN 1 ELSE 0 
            END AS is_admin_drag_task
        FROM raw_tasks
    )
    SELECT 
        soc_code,
        ROUND(AVG(CASE WHEN is_augmentation_task = 1 THEN data_value END), 2) AS augmentation_score,
        ROUND(AVG(CASE WHEN is_admin_drag_task = 1 THEN data_value END), 2) AS admin_drag_score,
        ROUND(AVG(data_value), 2) AS overall_task_intensity
    FROM flagged_tasks
    GROUP BY soc_code
    HAVING augmentation_score IS NOT NULL AND admin_drag_score IS NOT NULL;
""")

parquet_task_path = os.path.join(SILVER_DIR, "fact_labor_task_decomposition.parquet").replace("\\", "/")
con.execute(f"COPY silver.fact_labor_task_decomposition TO '{parquet_task_path}' (FORMAT PARQUET);")
print("Created silver.fact_labor_task_decomposition")


Created silver.fact_labor_task_decomposition


## 5. Silver Layer Validation & Connection Teardown

In [6]:
# Validate Silver tables
print("=== Silver Layer Validation ===")
tables = [
    "fact_macro_monthly",
    "dim_labor_occupations",
    "fact_tech_investment",
    "fact_labor_task_decomposition"
]
for t in tables:
    count = con.execute(f"SELECT COUNT(*) FROM silver.{t}").fetchone()[0]
    print(f"silver.{t:<30}: {count:>6} rows")


=== Silver Layer Validation ===
silver.fact_macro_monthly            :    140 rows
silver.dim_labor_occupations         :    696 rows
silver.fact_tech_investment          :    106 rows
silver.fact_labor_task_decomposition :    786 rows


In [7]:
# Explicitly close DuckDB connection
con.close()
print("DuckDB connection successfully closed.")


DuckDB connection successfully closed.
